# Python vs. Fortran - Core Differences and Fundamentals in HPC

## Introduction
The transition from traditional compiled languages like Fortran to Python in High-Performance Computing (HPC) involves a fundamental trade-off: **Developer Time (Time-to-Science) versus Execution Time (Time-to-Solution)**. Fortran was designed for raw numerical performance and static memory allocation. Python, on the other hand, is a high-level, general-purpose language that prioritizes readability, rapid prototyping, and ecosystem integration. 

In this notebook, we explore the architectural differences between Python and Fortran, demonstrating why Python is widely adopted as a "glue language" in modern HPC, and highlighting the performance bottlenecks that dictate how we must structure parallel codes.

## 1. Compilation vs. Interpretation

The most defining difference between Fortran and Python is how they are processed by the machine.

* **Fortran (Compiled Ahead-of-Time):** Fortran code is translated by a compiler (like `gfortran` or `ifort`) directly into highly optimized machine code specifically tailored for the target CPU architecture before execution. The compiler analyzes the entire program, performs loop unrolling, and optimizes memory access. During execution, there is zero translation overhead.
* **Python (Interpreted at Runtime):** Python compiles source code into intermediate *bytecode* at runtime, which is then executed by the Python Virtual Machine (PVM). This step-by-step interpretation introduces significant overhead. 

Let's observe Python's bytecode compilation in action using the built-in `dis` (disassembler) module.

In [1]:
import dis

def compute_energy(mass, velocity):
    # Simple kinetic energy calculation
    return 0.5 * mass * (velocity ** 2)

# Disassembling the Python function to see the bytecode instructions
print("Python Bytecode Instructions for compute_energy:\n")
dis.dis(compute_energy)

Python Bytecode Instructions for compute_energy:

  5           0 LOAD_CONST               1 (0.5)
              2 LOAD_FAST                0 (mass)
              4 BINARY_MULTIPLY
              6 LOAD_FAST                1 (velocity)
              8 LOAD_CONST               2 (2)
             10 BINARY_POWER
             12 BINARY_MULTIPLY
             14 RETURN_VALUE


**The Fortran Perspective:**
In Fortran, the equivalent calculation is reduced to native CPU instructions (like `MUL`, `ADD` registers) during the compile step.

real function compute_energy(mass, velocity)
    real, intent(in) :: mass, velocity
    compute_energy = 0.5 * mass * (velocity ** 2)
end function compute_energy

While Python evaluates the type and operation at runtime via the bytecode (BINARY_MULTIPLY, BINARY_POWER), Fortran already knows the types and memory addresses, resulting in vastly superior execution speed for basic loops.

## 2. Dynamic vs. Static Typing

Another critical distinction is how variables are managed in memory.

* **Fortran (Static Typing):** Variable types must be explicitly declared before compilation. Once compiled, a variable strictly holds that data type, and its memory footprint is fixed.
* **Python (Dynamic Typing):** Python binds names to objects at runtime. A variable can point to an integer, and in the next line, point to a list or a string. While this provides massive flexibility for writing generic code, it requires constant runtime type-checking. Every time Python adds two numbers, it must first inspect both objects to verify they support the addition operation.

Let's demonstrate Python's dynamic nature and how it handles heterogeneous data structures effortlessly compared to Fortran.

In [2]:
# In Python, variables are dynamically bound and can change type on the fly
x = 100          # x is an integer
print(f"Type of x: {type(x)} | Value: {x}")

x = [1.5, 2.5]   # x is now a list of floats
print(f"Type of x: {type(x)} | Value: {x}")

# Python effortlessly stores heterogeneous data (lists containing mixed types)
# Doing this in Fortran would require complex derived types or multiple arrays.
particle_data = ["Electron", 9.109e-31, -1.602e-19, True]

print(f"\nParticle Data: {particle_data}")
for item in particle_data:
    print(f"Data: {item} | Type: {type(item)}")

Type of x: <class 'int'> | Value: 100
Type of x: <class 'list'> | Value: [1.5, 2.5]

Particle Data: ['Electron', 9.109e-31, -1.602e-19, True]
Data: Electron | Type: <class 'str'>
Data: 9.109e-31 | Type: <class 'float'>
Data: -1.602e-19 | Type: <class 'float'>
Data: True | Type: <class 'bool'>


**The Fortran Perspective:**
To handle the heterogeneous `particle_data` in Fortran, one cannot simply use an array (which must be strictly homogeneous). A `type` definition is required:

type :: Particle
    character(len=20) :: name
    real(8) :: mass
    real(8) :: charge
    logical :: is_active
end type Particle

type(Particle) :: electron
electron%name = "Electron"
electron%mass = 9.109e-31

Python's dynamic typing allows for much faster setup and parsing of simulation parameters, but it is the exact reason why native Python loops over numerical arrays are slow, necessitating tools like NumPy.

## 3. The Global Interpreter Lock (GIL)

In High-Performance Computing, leveraging multiple cores is mandatory. However, the standard implementation of Python has a built-in mechanism called the **Global Interpreter Lock (GIL)**. 

The GIL is a lock that allows only **one native thread to execute Python bytecodes at a time**. It exists because Python's memory management (reference counting) is not thread-safe. While the GIL makes writing single-threaded code easier and faster, it severely cripples multithreading for CPU-bound tasks. 

Let's demonstrate this by running a heavy mathematical computation. We will compare executing it sequentially versus dividing the work across two threads.

In [3]:
import time
import threading

def cpu_heavy_task(n):
    # A meaningless but CPU-intensive loop
    total = 0
    for i in range(n):
        total += i**2
    return total

n_iterations = 20_000_000

# 1. Sequential Execution
start = time.time()
cpu_heavy_task(n_iterations)
cpu_heavy_task(n_iterations)
seq_time = time.time() - start
print(f"Sequential execution time: {seq_time:.4f} seconds")

# 2. Multithreaded Execution (2 threads)
start = time.time()
thread1 = threading.Thread(target=cpu_heavy_task, args=(n_iterations,))
thread2 = threading.Thread(target=cpu_heavy_task, args=(n_iterations,))

thread1.start()
thread2.start()

thread1.join()
thread2.join()
thread_time = time.time() - start
print(f"Multithreaded execution time: {thread_time:.4f} seconds")

Sequential execution time: 29.1544 seconds
Multithreaded execution time: 43.4361 seconds


*Notice the output above:* The multithreaded version is not faster (and might even be slightly slower due to context switching overhead). The GIL forces the threads to wait in line, utilizing only a single CPU core despite having two threads spawned.

**The Fortran Perspective:**
Fortran does not have a GIL. Using standards like **OpenMP**, a Fortran programmer can easily spawn true OS-level threads that execute mathematical loops in parallel on shared memory, utilizing 100% of the available CPU cores automatically.

**The Python HPC Solution:**
Because standard threading is ineffective for CPU-bound simulations in Python, HPC developers must use **Multiprocessing** or **MPI (mpi4py)**. These approaches spawn completely separate, independent OS processes. Each process gets its own memory space and its own Python interpreter (with its own GIL), thus achieving true parallelism.

## 4. Memory Overhead: The Python Object Model

Memory footprint and data locality (how close data is in physical RAM) dictate performance in modern CPU architecture due to cache efficiency. 

* **Fortran Memory:** In Fortran, an integer is just a raw binary value. A 32-bit integer takes exactly 4 bytes of memory. An array of 1,000 integers takes exactly 4,000 contiguous bytes.
* **Python Memory:** Python follows the "everything is an object" philosophy. A simple integer in Python is actually a complex C-structure (`PyObject`). It contains the raw value, but also a reference count (for garbage collection) and a pointer to the type object. 

Let's inspect the real memory size of a basic Python variable.

In [4]:
import sys

# Standard Python integer vs List
python_int = 42
python_list = [1, 2, 3, 4, 5]

print(f"Size of a single Python integer: {sys.getsizeof(python_int)} bytes")
print(f"Size of an empty Python list: {sys.getsizeof([])} bytes")
print(f"Size of a Python list with 5 ints: {sys.getsizeof(python_list)} bytes\n")

# To show that lists do not store raw numbers, but pointers to objects:
print("Memory addresses of elements in the list:")
for i, item in enumerate(python_list):
    print(f"Index {i}: Object Address = {id(item)}")

Size of a single Python integer: 28 bytes
Size of an empty Python list: 56 bytes
Size of a Python list with 5 ints: 104 bytes

Memory addresses of elements in the list:
Index 0: Object Address = 140643710484720
Index 1: Object Address = 140643710484752
Index 2: Object Address = 140643710484784
Index 3: Object Address = 140643710484816
Index 4: Object Address = 140643710484848


**The Fortran Perspective vs. The Python Reality:**

As seen above, a single integer in Python takes 28 bytes (on a 64-bit system), which is 7 times larger than a Fortran 4-byte integer. Furthermore, a Python `list` does not store the actual numbers sequentially in memory. It stores an array of *pointers* scattered across the RAM, pointing to integer objects. 

This fragmentation destroys **CPU Cache Spatial Locality**. When a CPU fetches data, it loads chunks of contiguous memory. Fortran arrays perfectly fit into CPU caches, allowing vectorization (SIMD). Python lists cause constant cache misses, severely degrading performance in nested physical analysis loops.

This is the reason why external libraries like **NumPy** are mandatory when complex calculations are required. NumPy bypasses the Python object model entirely by allocating raw, contiguous, Fortran-like memory blocks accessed via compiled C code, bringing HPC performance back to Python.